In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected")

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [2]:
!python --version
!which nvidia-smi
!ls /usr/local/cuda/bin/nvidia-smi

Python 3.13.15
/opt/bin/nvidia-smi
ls: cannot access '/usr/local/cuda/bin/nvidia-smi': No such file or directory


In [3]:
!nvidia-smi
!git clone https://github.com/AhanaSrinivas/legal-text-classification.git
%cd legal-text-classification
!git rev-parse HEAD
!pip install -q transformers datasets scikit-learn psutil huggingface_hub
!python scripts/04_train_transformer.py --epochs 3 --batch_size 8 --gradient_accumulation_steps 2 --results_dir results_run2 --output_dir models/transformer

Sun Oct  4 03:04:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
import json, torch
from google.colab import userdata
from huggingface_hub import HfApi

info = json.load(open('results_run2/transformer_run_info.json'))

print(info['smoke_test'], info['epochs'], info['device'], info['test_macro_f1'])

assert info['smoke_test'] is False and info['epochs'] == 3

ck = torch.load(
    'models/transformer/best_model.pt',
    map_location='cpu'
)

torch.save(
    ck['model_state_dict'],
    'legal_bert_state.pt'
)

HF_TOKEN = userdata.get('HF_TOKEN').strip()
api = HfApi(token=HF_TOKEN)

repo = "AhanaSrinivas/legal-text-classification-artifacts"

api.upload_file(
    path_or_fileobj='legal_bert_state.pt',
    path_in_repo='legal_bert_state.pt',
    repo_id=repo
)

api.upload_file(
    path_or_fileobj='results_run2/transformer_run_info.json',
    path_in_repo='transformer_run2_info.json',
    repo_id=repo
)

print("uploaded")

False 3 cuda 0.5319059168277114


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ation/legal_bert_state.pt:   0%|          |  567kB /  438MB            

uploaded
